In [3]:
# ab_test_analysis.py
# Complete A/B test analysis for your project
# Run:
#   python ab_test_analysis.py
#
# Required packages:
#   pandas numpy matplotlib scipy statsmodels

import os
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.stats import mannwhitneyu
from statsmodels.stats.proportion import (
    proportions_ztest,
    proportion_confint,
    confint_proportions_2indep
)
from statsmodels.stats.power import NormalIndPower
import statsmodels.api as sm
import statsmodels.formula.api as smf

warnings.filterwarnings("ignore")

# ============================================================
# 0. CONFIG
# ============================================================
USERS_PATH = "users_rows.csv"
EVENTS_PATH = "events_rows.csv"
OUTPUT_DIR = "abtest_outputs"

ALPHA = 0.05
TARGET_POWER = 0.80

os.makedirs(OUTPUT_DIR, exist_ok=True)

# ============================================================
# 1. LOAD DATA
# ============================================================
users = pd.read_csv(USERS_PATH)
events = pd.read_csv(EVENTS_PATH)

users["assigned_at"] = pd.to_datetime(users["assigned_at"], errors="coerce")
events["timestamp"] = pd.to_datetime(events["timestamp"], errors="coerce")

# merge group into events
events = events.merge(users[["user_id", "group"]], on="user_id", how="left")

print("\n==============================")
print("DATA LOADED")
print("==============================")
print(f"Users shape:  {users.shape}")
print(f"Events shape: {events.shape}")
print("\nUsers columns:", users.columns.tolist())
print("Events columns:", events.columns.tolist())

# ============================================================
# 2. BASIC DATA SUMMARY
# ============================================================
group_counts = users["group"].value_counts().sort_index()
print("\n==============================")
print("GROUP COUNTS")
print("==============================")
print(group_counts)

event_counts = events["event_type"].value_counts()
print("\n==============================")
print("EVENT TYPE COUNTS")
print("==============================")
print(event_counts)

# ============================================================
# 3. BUILD USER-LEVEL METRICS
# ============================================================
# total clicks per user
user_clicks = (
    events.loc[events["event_type"] == "click"]
    .groupby("user_id")
    .size()
    .rename("total_clicks")
)

# clicked at least once
user_clicked_binary = (
    events.loc[events["event_type"] == "click"]
    .groupby("user_id")
    .size()
    .gt(0)
    .astype(int)
    .rename("clicked_any")
)

# avg article time per user
user_article_time = (
    events.loc[events["event_type"] == "article_time"]
    .groupby("user_id")["value"]
    .mean()
    .rename("avg_article_time")
)

# max scroll depth per user
user_scroll = (
    events.loc[events["event_type"] == "scroll"]
    .groupby("user_id")["value"]
    .max()
    .rename("max_scroll_depth")
)

# session duration per user
user_session = (
    events.loc[events["event_type"] == "session_end"]
    .groupby("user_id")["value"]
    .max()
    .rename("session_duration")
)

# article impressions approximated as number of unique article ids seen/interacted with
user_article_impressions = (
    events.loc[events["article_id"].notna()]
    .groupby("user_id")["article_id"]
    .nunique()
    .rename("card_impressions")
)

user_metrics = users.copy()
user_metrics = user_metrics.merge(user_clicks, on="user_id", how="left")
user_metrics = user_metrics.merge(user_clicked_binary, on="user_id", how="left")
user_metrics = user_metrics.merge(user_article_time, on="user_id", how="left")
user_metrics = user_metrics.merge(user_scroll, on="user_id", how="left")
user_metrics = user_metrics.merge(user_session, on="user_id", how="left")
user_metrics = user_metrics.merge(user_article_impressions, on="user_id", how="left")

user_metrics["total_clicks"] = user_metrics["total_clicks"].fillna(0).astype(int)
user_metrics["clicked_any"] = user_metrics["clicked_any"].fillna(0).astype(int)
user_metrics["card_impressions"] = user_metrics["card_impressions"].fillna(0).astype(int)

# user-level CTR = clicks / article opportunities actually observed
user_metrics["ctr_observed"] = np.where(
    user_metrics["card_impressions"] > 0,
    user_metrics["total_clicks"] / user_metrics["card_impressions"],
    np.nan
)

# ============================================================
# 4. PRIMARY CTR ANALYSIS
# ============================================================
# We define the primary CTR as click opportunities across all articles.
# Since your data contain 6 distinct article IDs, we can use:
#   opportunities = number of users * number of unique articles
# This mirrors the stronger version of your friend's analysis.

n_articles = events["article_id"].dropna().nunique()
n_a = (users["group"] == "A").sum()
n_b = (users["group"] == "B").sum()

clicks_a = user_metrics.loc[user_metrics["group"] == "A", "total_clicks"].sum()
clicks_b = user_metrics.loc[user_metrics["group"] == "B", "total_clicks"].sum()

opportunities_a = n_a * n_articles
opportunities_b = n_b * n_articles

ctr_a = clicks_a / opportunities_a if opportunities_a > 0 else np.nan
ctr_b = clicks_b / opportunities_b if opportunities_b > 0 else np.nan
ctr_diff = ctr_b - ctr_a

z_stat, p_value_ctr = proportions_ztest(
    count=np.array([clicks_a, clicks_b]),
    nobs=np.array([opportunities_a, opportunities_b]),
    alternative="two-sided"
)

# per-group binomial confidence intervals
ci_a_low, ci_a_high = proportion_confint(clicks_a, opportunities_a, alpha=ALPHA, method="wilson")
ci_b_low, ci_b_high = proportion_confint(clicks_b, opportunities_b, alpha=ALPHA, method="wilson")

# confidence interval for difference in proportions
try:
    diff_ci_low, diff_ci_high = confint_proportions_2indep(
        count1=clicks_b, nobs1=opportunities_b,
        count2=clicks_a, nobs2=opportunities_a,
        method="wald"
    )
except Exception:
    diff_ci_low, diff_ci_high = np.nan, np.nan

ctr_table = pd.DataFrame({
    "group": ["A", "B"],
    "clicks": [clicks_a, clicks_b],
    "opportunities": [opportunities_a, opportunities_b],
    "CTR": [ctr_a, ctr_b],
    "CI_low": [ci_a_low, ci_b_low],
    "CI_high": [ci_a_high, ci_b_high]
})

print("\n==============================")
print("PRIMARY ANALYSIS: IMPRESSION-BASED CTR")
print("==============================")
print(ctr_table.to_string(index=False))
print(f"\nCTR difference (B - A): {ctr_diff:.4f}")
print(f"Z-statistic: {z_stat:.4f}")
print(f"p-value:     {p_value_ctr:.4f}")
print(f"95% CI for difference (B - A): [{diff_ci_low:.4f}, {diff_ci_high:.4f}]")

if p_value_ctr < ALPHA:
    print("Result: statistically significant difference in CTR.")
else:
    print("Result: NOT statistically significant difference in CTR.")

# ============================================================
# 5. SUPPLEMENTARY USER-LEVEL CLICK TEST
# ============================================================
click_any_summary = (
    user_metrics.groupby("group")["clicked_any"]
    .agg(["sum", "count"])
    .rename(columns={"sum": "clicked_users", "count": "users"})
)
click_any_summary["rate"] = click_any_summary["clicked_users"] / click_any_summary["users"]

z_user, p_user = proportions_ztest(
    count=click_any_summary["clicked_users"].values,
    nobs=click_any_summary["users"].values,
    alternative="two-sided"
)

print("\n==============================")
print("SUPPLEMENTARY: USER-LEVEL CLICK TEST")
print("==============================")
print(click_any_summary)
print(f"\nZ-statistic: {z_user:.4f}")
print(f"p-value:     {p_user:.4f}")

# ============================================================
# 6. SECONDARY METRICS: MANN-WHITNEY U TESTS
# ============================================================
def run_mann_whitney(df, variable, label):
    a = df.loc[df["group"] == "A", variable].dropna()
    b = df.loc[df["group"] == "B", variable].dropna()

    print("\n------------------------------")
    print(label)
    print("------------------------------")
    print(f"n_A = {len(a)}, n_B = {len(b)}")

    if len(a) == 0 or len(b) == 0:
        print("Insufficient data.")
        return None

    stat, p = mannwhitneyu(a, b, alternative="two-sided")
    print(f"Median A: {np.median(a):.3f}")
    print(f"Median B: {np.median(b):.3f}")
    print(f"U-stat:   {stat:.3f}")
    print(f"p-value:  {p:.4f}")
    return {"metric": variable, "u_stat": stat, "p_value": p}

print("\n==============================")
print("SECONDARY METRICS")
print("==============================")

secondary_results = []
for var, label in [
    ("avg_article_time", "Average Article Read Time"),
    ("max_scroll_depth", "Maximum Scroll Depth"),
    ("session_duration", "Session Duration"),
    ("ctr_observed", "Observed User-Level CTR"),
]:
    out = run_mann_whitney(user_metrics, var, label)
    if out is not None:
        secondary_results.append(out)

secondary_results_df = pd.DataFrame(secondary_results)

# ============================================================
# 7. ARTICLE-LEVEL PANEL FOR REPEATED-MEASURES MODEL
# ============================================================
# We create one row per user x article, with clicked = 1 if clicked at least once.
# This captures repeated observations within each user.
article_map = (
    events.loc[events["article_id"].notna(), ["article_id", "article_position"]]
    .drop_duplicates()
    .sort_values(["article_position", "article_id"])
)

# full user x article grid
user_article_panel = (
    users[["user_id", "group"]]
    .assign(key=1)
    .merge(article_map.assign(key=1), on="key", how="outer")
    .drop(columns="key")
)

# whether clicked at least once for each user/article
click_binary = (
    events.loc[(events["event_type"] == "click") & (events["article_id"].notna())]
    .groupby(["user_id", "article_id"])
    .size()
    .gt(0)
    .astype(int)
    .rename("clicked")
    .reset_index()
)

panel = user_article_panel.merge(click_binary, on=["user_id", "article_id"], how="left")
panel["clicked"] = panel["clicked"].fillna(0).astype(int)

# article-level time, if useful
article_time_panel = (
    events.loc[(events["event_type"] == "article_time") & (events["article_id"].notna())]
    .groupby(["user_id", "article_id"])["value"]
    .mean()
    .rename("article_time")
    .reset_index()
)
panel = panel.merge(article_time_panel, on=["user_id", "article_id"], how="left")

# ============================================================
# 8. ARTICLE-LEVEL MODEL
# ============================================================
# Directly runnable and stable choice:
# GEE Binomial with clustering by user_id
# This handles repeated article observations within the same user.
print("\n==============================")
print("ARTICLE-LEVEL MODEL (GEE BINOMIAL)")
print("==============================")

panel["group_B"] = (panel["group"] == "B").astype(int)

gee_ok = True
try:
    gee_model = smf.gee(
        "clicked ~ group_B + article_position",
        groups="user_id",
        data=panel,
        family=sm.families.Binomial(),
        cov_struct=sm.cov_struct.Exchangeable()
    ).fit()

    print(gee_model.summary())

    coef = gee_model.params.get("group_B", np.nan)
    se = gee_model.bse.get("group_B", np.nan)
    p_gee = gee_model.pvalues.get("group_B", np.nan)
    or_gee = np.exp(coef)
    ci_low_gee = np.exp(coef - 1.96 * se)
    ci_high_gee = np.exp(coef + 1.96 * se)

    print(f"\nGroup B odds ratio: {or_gee:.3f}")
    print(f"95% CI: [{ci_low_gee:.3f}, {ci_high_gee:.3f}]")
    print(f"p-value: {p_gee:.4f}")

except Exception as e:
    gee_ok = False
    print("GEE model failed.")
    print("Reason:", str(e))

# ============================================================
# 9. POWER ANALYSIS
# ============================================================
print("\n==============================")
print("POWER ANALYSIS")
print("==============================")

analysis = NormalIndPower()
n_per_group = min(n_a, n_b)

# observed effect size for CTR
# Cohen's h = 2*asin(sqrt(p1)) - 2*asin(sqrt(p2))
def cohens_h(p1, p2):
    p1 = np.clip(p1, 1e-8, 1 - 1e-8)
    p2 = np.clip(p2, 1e-8, 1 - 1e-8)
    return 2 * np.arcsin(np.sqrt(p1)) - 2 * np.arcsin(np.sqrt(p2))

h_observed = abs(cohens_h(ctr_a, ctr_b))

try:
    achieved_power = analysis.power(
        effect_size=h_observed,
        nobs1=n_per_group,
        alpha=ALPHA,
        ratio=1.0,
        alternative="two-sided"
    )
except Exception:
    achieved_power = np.nan

try:
    mde_h = analysis.solve_power(
        power=TARGET_POWER,
        nobs1=n_per_group,
        alpha=ALPHA,
        ratio=1.0,
        alternative="two-sided"
    )
except Exception:
    mde_h = np.nan

print(f"n per group (approx): {n_per_group}")
print(f"Observed CTR_A: {ctr_a:.4f}")
print(f"Observed CTR_B: {ctr_b:.4f}")
print(f"Observed Cohen's h: {h_observed:.4f}")
print(f"Achieved power (using observed h): {achieved_power:.4f}")
print(f"Minimum detectable effect size h at 80% power: {mde_h:.4f}")

# ============================================================
# 10. VISUALIZATIONS
# ============================================================
print("\n==============================")
print("GENERATING PLOTS")
print("==============================")

# ---------- Plot 1: CTR bar chart with CI ----------
fig, ax = plt.subplots(figsize=(7, 5))
x = np.arange(2)
ctr_vals = ctr_table["CTR"].values
lower_err = ctr_vals - ctr_table["CI_low"].values
upper_err = ctr_table["CI_high"].values - ctr_vals

ax.bar(x, ctr_vals)
ax.errorbar(
    x, ctr_vals,
    yerr=[lower_err, upper_err],
    fmt="none",
    capsize=6
)
ax.set_xticks(x)
ax.set_xticklabels(["A", "B"])
ax.set_ylabel("CTR")
ax.set_title("Impression-Based CTR by Group")
ax.text(
    0.5, max(ctr_vals) + 0.02,
    f"p = {p_value_ctr:.4f}",
    ha="center"
)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "ctr_by_group.png"), dpi=150)
plt.close()

# ---------- Plot 2: total clicks distribution ----------
fig, ax = plt.subplots(figsize=(7, 5))
for g in ["A", "B"]:
    vals = user_metrics.loc[user_metrics["group"] == g, "total_clicks"]
    ax.hist(vals, bins=np.arange(vals.max() + 3) - 0.5, alpha=0.7, label=g)
ax.set_xlabel("Total Clicks per User")
ax.set_ylabel("Count")
ax.set_title("Distribution of Total Clicks per User")
ax.legend()
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "click_distribution.png"), dpi=150)
plt.close()

# ---------- Plot 3: article time by group ----------
tmp = user_metrics.dropna(subset=["avg_article_time"])
if len(tmp) > 0:
    fig, ax = plt.subplots(figsize=(7, 5))
    data_to_plot = [
        tmp.loc[tmp["group"] == "A", "avg_article_time"].dropna().values,
        tmp.loc[tmp["group"] == "B", "avg_article_time"].dropna().values
    ]
    ax.boxplot(data_to_plot, tick_labels=["A", "B"])
    ax.set_ylabel("Seconds")
    ax.set_title("Average Article Read Time by Group")
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, "article_time_boxplot.png"), dpi=150)
    plt.close()

# ---------- Plot 4: scroll depth by group ----------
tmp = user_metrics.dropna(subset=["max_scroll_depth"])
if len(tmp) > 0:
    fig, ax = plt.subplots(figsize=(7, 5))
    data_to_plot = [
        tmp.loc[tmp["group"] == "A", "max_scroll_depth"].dropna().values,
        tmp.loc[tmp["group"] == "B", "max_scroll_depth"].dropna().values
    ]
    ax.boxplot(data_to_plot, tick_labels=["A", "B"])
    ax.set_ylabel("Max Scroll Depth")
    ax.set_title("Maximum Scroll Depth by Group")
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, "scroll_depth_boxplot.png"), dpi=150)
    plt.close()

# ---------- Plot 5: session duration by group ----------
tmp = user_metrics.dropna(subset=["session_duration"])
if len(tmp) > 0:
    fig, ax = plt.subplots(figsize=(7, 5))
    data_to_plot = [
        tmp.loc[tmp["group"] == "A", "session_duration"].dropna().values,
        tmp.loc[tmp["group"] == "B", "session_duration"].dropna().values
    ]
    ax.boxplot(data_to_plot, tick_labels=["A", "B"])
    ax.set_ylabel("Seconds")
    ax.set_title("Session Duration by Group")
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, "session_duration_boxplot.png"), dpi=150)
    plt.close()

# ---------- Plot 6: article-level click heatmap-like table ----------
heat = (
    panel.groupby(["article_position", "group"])["clicked"]
    .mean()
    .unstack()
    .sort_index()
)

if not heat.empty:
    fig, ax = plt.subplots(figsize=(6, 5))
    im = ax.imshow(heat.values, aspect="auto")
    ax.set_xticks(np.arange(len(heat.columns)))
    ax.set_xticklabels(heat.columns)
    ax.set_yticks(np.arange(len(heat.index)))
    ax.set_yticklabels(heat.index.astype(int))
    ax.set_xlabel("Group")
    ax.set_ylabel("Article Position")
    ax.set_title("Click Rate by Article Position and Group")
    for i in range(heat.shape[0]):
        for j in range(heat.shape[1]):
            ax.text(j, i, f"{heat.iloc[i, j]:.2f}", ha="center", va="center")
    plt.colorbar(im, ax=ax, label="Click Rate")
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, "click_heatmap.png"), dpi=150)
    plt.close()

# ---------- Plot 7: power curve ----------
n_range = np.arange(10, 101, 5)
effect_sizes = [0.2, 0.3, 0.5, 0.8]

fig, ax = plt.subplots(figsize=(8, 5))
for h in effect_sizes:
    powers = []
    for n in n_range:
        try:
            p = analysis.power(effect_size=h, nobs1=n, alpha=ALPHA, ratio=1.0, alternative="two-sided")
        except Exception:
            p = np.nan
        powers.append(p)
    ax.plot(n_range, powers, label=f"h = {h}")

ax.axhline(TARGET_POWER, linestyle="--")
ax.axvline(n_per_group, linestyle=":")
ax.set_xlabel("Sample Size per Group")
ax.set_ylabel("Power")
ax.set_title("Power Curves for Two-Proportion Test")
ax.legend()
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "power_curve.png"), dpi=150)
plt.close()

# ============================================================
# 11. SAVE TABLES
# ============================================================
ctr_table.to_csv(os.path.join(OUTPUT_DIR, "ctr_table.csv"), index=False)
user_metrics.to_csv(os.path.join(OUTPUT_DIR, "user_metrics.csv"), index=False)
panel.to_csv(os.path.join(OUTPUT_DIR, "article_level_panel.csv"), index=False)

if len(secondary_results_df) > 0:
    secondary_results_df.to_csv(os.path.join(OUTPUT_DIR, "secondary_tests.csv"), index=False)

# ============================================================
# 12. FINAL INTERPRETATION
# ============================================================
print("\n==============================")
print("FINAL INTERPRETATION")
print("==============================")
print(f"Primary CTR p-value: {p_value_ctr:.4f}")
print(f"User-level click p-value: {p_user:.4f}")

if gee_ok:
    print("Article-level repeated-measures model: fitted successfully.")
else:
    print("Article-level repeated-measures model: not available for this run.")

print("\nReminder:")
print("- Primary CTR test is your main inferential result.")
print("- Secondary metrics are exploratory.")
print("- If you discuss many secondary p-values, mention multiple-comparison caution.")
print("- With a small sample, low power is a likely reason for non-significant findings.")

print(f"\nOutputs saved to: {OUTPUT_DIR}")
print("Done.")


DATA LOADED
Users shape:  (30, 4)
Events shape: (100, 8)

Users columns: ['user_id', 'group', 'assigned_at', 'user_agent']
Events columns: ['event_id', 'user_id', 'event_type', 'article_id', 'article_position', 'value', 'timestamp', 'group']

GROUP COUNTS
group
A    16
B    14
Name: count, dtype: int64

EVENT TYPE COUNTS
event_type
scroll          48
page_view       19
click           13
article_time    12
session_end      8
Name: count, dtype: int64

PRIMARY ANALYSIS: IMPRESSION-BASED CTR
group  clicks  opportunities      CTR   CI_low  CI_high
    A      10             96 0.104167 0.057572 0.181222
    B       3             84 0.035714 0.012220 0.099817

CTR difference (B - A): -0.0685
Z-statistic: 1.7700
p-value:     0.0767
95% CI for difference (B - A): [-0.1413, 0.0044]
Result: NOT statistically significant difference in CTR.

SUPPLEMENTARY: USER-LEVEL CLICK TEST
       clicked_users  users      rate
group                                
A                  3     16  0.187500
B    